# B5 — Selection-atlas sweep overlap at Bouaké DE ABC / OBP loci

For each Bouaké DE ABC (gambiae) and OBP (coluzzii), check whether the gene coordinates fall inside any H12 sweep signal from the malariagen selection atlas (`h12-signal-detection-all.csv`). Ag3 data covers wild West African cohorts; Bouaké 2019 samples aren't in Ag3 themselves, so this tells us whether these loci are under selection more generally in the wild.

In [1]:
import os
os.chdir('/Users/sanjay.nagi/Data-Science/bouake')
import numpy as np, pandas as pd
import plotly.graph_objects as go
import malariagen_data

# Load sweep signals
signals = pd.read_csv('skills/selection-atlas/h12-signal-detection-all.csv')
signals['taxon'] = signals['cohort_id'].str.extract(r'_(gamb|colu|arab|bissau|fun)_')
signals['country'] = signals['cohort_id'].str[:2]
print('signals shape:', signals.shape)
print(signals['taxon'].value_counts())

# Load Bouaké DE tables (B1 output)
abc_gam = pd.read_csv('results/abc_obp/bouake_de_abc_gambiae.csv')
obp_col = pd.read_csv('results/abc_obp/bouake_de_obp_coluzzii.csv')
print('ABC gambiae:', abc_gam.shape, ' OBP coluzzii:', obp_col.shape)

signals shape: (357, 32)
taxon
colu    178
gamb    145
arab     17
Name: count, dtype: int64
ABC gambiae: (99, 16)  OBP coluzzii: (74, 16)


In [2]:
# Load Ag3 geneset and build gene coordinate lookup
ag3 = malariagen_data.Ag3(results_cache='~/ag3_cache', pre=True)
gs = ag3.geneset()
genes = gs[gs['type'] == 'gene'][['contig','start','end','ID','Name','description']].rename(columns={'ID':'GeneID'})
print('gene count:', len(genes))

# Convert Ag3 coords (2L/2R/3L/3R/X) into compound 2RL/3RL used by selection atlas.
# Atlas contigs: 2RL = 2R + 2L (2R first), 3RL = 3R + 3L (3R first), X = X.
# Offsets from Ag3 genome_sequence:
OFFSETS = {'2R': 0, '2L': 61545105, '3R': 0, '3L': 53200684, 'X': 0}
ARM_TO_CONTIG = {'2R': '2RL', '2L': '2RL', '3R': '3RL', '3L': '3RL', 'X': 'X'}

def to_atlas_coords(contig, start, end):
    if contig not in ARM_TO_CONTIG:
        return None, None, None
    return ARM_TO_CONTIG[contig], start + OFFSETS[contig], end + OFFSETS[contig]

coords = genes.apply(lambda r: to_atlas_coords(r['contig'], r['start'], r['end']), axis=1, result_type='expand')
coords.columns = ['atlas_contig','atlas_start','atlas_end']
genes_atlas = pd.concat([genes, coords], axis=1)
print(genes_atlas.head(3))

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.09)

Load genome features: ⠹ (0:00:00.17)

Load genome features: ⠸ (0:00:00.26)

Load genome features: ⠼ (0:00:00.34)

...

Load genome features: ⠋ (0:00:01.70)

Load genome features: ⠙ (0:00:01.87)

Load genome features: ⠹ (0:00:01.98)

gene count: 13057


   contig   start     end      GeneID Name  \
1      2L  157348  186936  AGAP004677  NaN   
24     2L  203779  205293  AGAP004678  NaN   
32     2L  207894  210460  AGAP004679  NaN   

                                          description atlas_contig  \
1   methylenetetrahydrofolate dehydrogenase(NAD ) ...          2RL   
24                                                NaN          2RL   
32  Folylpolyglutamate synthase [Source:UniProtKB/...          2RL   

    atlas_start   atlas_end  
1    61702453.0  61732041.0  
24   61748884.0  61750398.0  
32   61752999.0  61755565.0  


In [3]:
# Overlap function — given a gene, return the sweep signals whose span1 interval overlaps it.
# Use span1 (conservative) rather than span2 (broad), as per skills/selection-atlas/signals.md.
WEST_AFRICA = {'CI','BF','GH','ML','TG','BJ','GN','GW','GM','SN','SL','LR','NG','NE','CM'}

def overlap_sweeps(gene_id, taxon_filter=None, region='west_africa'):
    g = genes_atlas[genes_atlas['GeneID'] == gene_id]
    if g.empty or g.iloc[0]['atlas_contig'] is None:
        return pd.DataFrame()
    r = g.iloc[0]
    s = signals[signals['contig'] == r['atlas_contig']].copy()
    s = s[(s['span1_pstart'] <= r['atlas_end']) & (s['span1_pstop'] >= r['atlas_start'])]
    if taxon_filter:
        s = s[s['taxon'].isin(taxon_filter)]
    if region == 'west_africa':
        s = s[s['country'].isin(WEST_AFRICA)]
    return s.sort_values('delta_i', ascending=False)

# Quick sanity: Vgsc (AGAP004707) should overlap known kdr sweeps
sanity = overlap_sweeps('AGAP004707', taxon_filter=['gamb','colu'])
print('Vgsc overlap sanity (n rows):', len(sanity))
print(sanity[['cohort_id','delta_i','stat_max','pcenter']].head(5).to_string(index=False))

Vgsc overlap sanity (n rows): 29
                    cohort_id  delta_i  stat_max  pcenter
     BF-09_Houet_colu_2012_Q3     4018  0.949540 64235752
     BF-09_Houet_colu_2014_Q3     3628  1.000000 64264735
        BF-02_Comoe_colu_2012     3601  0.750063 64242303
     BF-09_Houet_gamb_2012_Q3     3597  1.000000 64140094
GN-F_Kissidougou_gamb_2012_Q4     3492  0.961361 64106159


In [4]:
# For each Bouaké DE ABC (sig up, gambiae), tag with sweep overlap summary
def sweep_summary(gene_id, taxon_filter):
    s = overlap_sweeps(gene_id, taxon_filter=taxon_filter, region='west_africa')
    if s.empty:
        return pd.Series({'n_sweeps':0,'cohorts':'','max_delta_i':np.nan,'max_stat_max':np.nan,'countries':''})
    return pd.Series({
        'n_sweeps': len(s),
        'cohorts': '; '.join(s['cohort_id'].tolist()[:5]),
        'max_delta_i': s['delta_i'].max(),
        'max_stat_max': s['stat_max'].max(),
        'countries': ','.join(sorted(s['country'].unique())),
    })

abc_sig = abc_gam[abc_gam['padj'] < 0.05].sort_values('log2FoldChange', ascending=False).copy()
abc_sig[['n_sweeps','cohorts','max_delta_i','max_stat_max','countries']] = abc_sig['GeneID'].apply(lambda g: sweep_summary(g, ['gamb','colu']))

obp_sig = obp_col[obp_col['padj'] < 0.05].sort_values('log2FoldChange', ascending=False).copy()
obp_sig[['n_sweeps','cohorts','max_delta_i','max_stat_max','countries']] = obp_sig['GeneID'].apply(lambda g: sweep_summary(g, ['gamb','colu']))

print('ABC gambiae — genes with ≥1 West African sweep overlap:', int((abc_sig['n_sweeps']>0).sum()), 'of', len(abc_sig))
print('OBP coluzzii — genes with ≥1 West African sweep overlap:', int((obp_sig['n_sweeps']>0).sum()), 'of', len(obp_sig))

abc_sig.to_csv('results/abc_obp/sweep_overlap_abc_gambiae.csv', index=False)
obp_sig.to_csv('results/abc_obp/sweep_overlap_obp_coluzzii.csv', index=False)

print('\nABC genes with sweeps:')
print(abc_sig[abc_sig['n_sweeps']>0][['GeneID','GeneName','log2FoldChange','n_sweeps','max_delta_i','max_stat_max','countries']].to_string(index=False))
print('\nOBP genes with sweeps:')
print(obp_sig[obp_sig['n_sweeps']>0][['GeneID','GeneName','log2FoldChange','n_sweeps','max_delta_i','max_stat_max','countries']].to_string(index=False))

ABC gambiae — genes with ≥1 West African sweep overlap: 10 of 35
OBP coluzzii — genes with ≥1 West African sweep overlap: 1 of 17

ABC genes with sweeps:
    GeneID GeneName  log2FoldChange  n_sweeps  max_delta_i  max_stat_max            countries
AGAP008436   ABCC11        2.263626         1       1194.0      0.105859                   CI
AGAP008437    ABCC9        1.902601         1       1194.0      0.105859                   CI
AGAP027980      NaN        1.289861         1       1194.0      0.105859                   CI
AGAP002638    ABCH1        1.231517         8       3261.0      0.323129             BF,GH,ML
AGAP028128      NaN        0.927320         1       1194.0      0.105859                   CI
AGAP003680    ABCH2        0.542103         2       3071.0      0.140850                   CM
AGAP002858      NaN        0.542000        32       7614.0      0.921110 BF,BJ,CI,CM,GH,GN,ML
AGAP000523      NaN        0.512178         6       2286.0      0.947042             BF,CI,GH


In [5]:
# Figure: Bouaké DE vs sweep overlap — highlight genes under selection in wild West African pops
def de_sweep_plot(de_df, family, title, out_stub, width=1000, height=700):
    d = de_df.copy()
    d['label'] = np.where(d['GeneName'].isna(), d['GeneID'], d['GeneName'].astype(str))
    d['has_sweep'] = d['n_sweeps'] > 0
    d = d.dropna(subset=['log2FoldChange']).reindex(d['log2FoldChange'].abs().sort_values(ascending=False).index).head(25).sort_values('log2FoldChange')

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=d['label'], x=d['log2FoldChange'], orientation='h',
        marker=dict(
            color=d['log2FoldChange'], colorscale='RdBu_r', cmid=0,
            line=dict(color=np.where(d['has_sweep'],'black','lightgrey'),
                      width=np.where(d['has_sweep'],2.0,0.3)),
        ),
        customdata=d[['GeneID','padj','n_sweeps','max_delta_i','max_stat_max','countries']].values,
        hovertemplate=('<b>%{y}</b><br>log2FC: %{x:.2f}<br>padj: %{customdata[1]:.2e}<br>'
                       'sweeps: %{customdata[2]}<br>max ΔAIC: %{customdata[3]:.0f}<br>'
                       'max H12: %{customdata[4]:.2f}<br>countries: %{customdata[5]}<extra></extra>'),
    ))
    fig.add_vline(x=0, line_color='grey')
    fig.update_layout(
        title=title + '<br><sup>Black outline = ≥1 West African H12 sweep overlaps gene (span1 interval, same-species cohorts)</sup>',
        xaxis_title='log2 fold change (PM vs Cont)', yaxis_title=family,
        template='simple_white', width=width, height=height, showlegend=False,
    )
    fig.write_html(f'figures_ms/{out_stub}.html')
    fig.write_image(f'figures_ms/{out_stub}.png', scale=2)
    return fig

fig_abc = de_sweep_plot(abc_sig, 'ABC', 'Top Bouaké DE ABCs (gambiae) — sweep overlap in wild West African Ag3 cohorts', 'b5_abc_sweep_overlap')
fig_abc.show()

/var/folders/yd/tz1zj4195ybgmg97z4y695w40000gp/T/ipykernel_25259/3623625860.py:28: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(f'figures_ms/{out_stub}.png', scale=2)
I0423 21:26:08.965526 1155998 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


In [6]:
fig_obp = de_sweep_plot(obp_sig, 'OBP', 'Top Bouaké DE OBPs (coluzzii) — sweep overlap in wild West African Ag3 cohorts', 'b5_obp_sweep_overlap')
fig_obp.show()

/var/folders/yd/tz1zj4195ybgmg97z4y695w40000gp/T/ipykernel_25259/3623625860.py:28: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(f'figures_ms/{out_stub}.png', scale=2)
